# Deep Learning for Time Series — Exercises

Companion to the note [Deep Learning for Time Series](Deep%20Learning%20for%20Time%20Series.md).

Practise the framing of neural forecasting (global models, multi-step strategies, covariates) and compute receptive fields, pinball losses and window counts by hand, then implement sliding windows, RevIN-style normalisation, causal dilated convolutions, the pinball loss, a DLinear-style forecaster and an N-BEATS residual stack from scratch in NumPy.

**12 exercises** · 🧠 Concept ×3 · ✍️ By hand ×4 · 💻 Code ×5

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper first; then type your numbers into the cell to check them.
- 💻 *Code* exercises: replace `return None` / `None` with your implementation and run the cell; the checks print ✅, ❌ or ⏳ (not attempted).
- Every exercise has a folded 💡 **Hint** and ✅ **Solution**. Try for a few minutes before opening them.

In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))
rng = np.random.default_rng(11)


## Part A · Concepts

### Exercise 1 · Local or global?
*🧠 Concept · ★☆☆*

A supermarket chain has 30,000 products in 200 stores with 3 years of daily sales. A small bakery has one product line with 2 years of daily sales. For each, would you train one model per series (local) or one model on all series (global)? Name a sensible first model for each.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

How much data does each individual series have, and do series share patterns?

</details>

<details>
<summary><b>✅ Solution</b></summary>

**Supermarket:** global. Millions of related series share weekly/holiday/promotion patterns, and many products are short or new (cold start). Start with LightGBM on lag + calendar + promotion features, then try a global neural model (DeepAR, TFT, N-HiTS). **Bakery:** local. One short series: seasonal naive and ETS/ARIMA as baselines; deep learning would likely overfit.

</details>

### Exercise 2 · Recursive vs direct multi-step
*🧠 Concept · ★★☆*

You need a 24-hour-ahead forecast. Explain the recursive strategy and the direct (multi-output) strategy, one advantage of each, and why errors compound in only one of them.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

What does the model receive as input at step 2 in each strategy?

</details>

<details>
<summary><b>✅ Solution</b></summary>

**Recursive:** train a one-step model and feed predictions back as inputs for the next step; one simple model, any horizon, but step-2 inputs are *predictions*, so errors compound and the train/test input distribution differs. **Direct / multi-output:** predict all 24 values at once from the true history; no compounding and each horizon can have its own pattern, but more outputs to learn and no use of the forecasted intermediate values. Neural forecasters (N-BEATS, TFT, PatchTST) almost always use multi-output.

</details>

### Exercise 3 · Why DLinear was a wake-up call
*🧠 Concept · ★★☆*

In 2022 a single linear layer applied to trend/seasonal decompositions (DLinear) beat several published long-horizon Transformers. What does this say about evaluating neural forecasters, and which simple baselines should every forecasting paper or project report?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Think about what a strong baseline protects you from.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Complex models can win on paper against weak baselines while losing to simple, well-tuned ones; benchmarks were sometimes dominated by trend/seasonality that a linear model captures. Always report **seasonal naive**, a classical model (ETS/ARIMA/Theta), a **linear model on windows** and gradient boosting with lags, use **time-based splits**, and use scale-free metrics (MASE). PatchTST later showed attention *can* help once patching and channel independence are used, but only the comparison with strong baselines makes that claim credible.

</details>

## Part B · By hand

### Exercise 4 · TCN receptive field
*✍️ By hand · ★☆☆*

A temporal convolutional network has 6 causal layers with kernel size 2 and dilations 1, 2, 4, 8, 16, 32. (a) What is its receptive field? (b) How many layers with kernel size 3 and doubling dilations (1, 2, 4, …) do you need to see at least 168 steps (one week of hourly data)?

In [ ]:
rf_a = None
layers_b = None
check('(a) receptive field', rf_a, 64); check('(b) layers needed', layers_b, 7)

<details>
<summary><b>💡 Hint</b></summary>

Receptive field $=1+(k-1)\sum_\ell d_\ell$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $1+1\cdot(1+2+4+8+16+32)=64$. (b) With $k=3$: $1+2(2^L-1)\ge168 \Rightarrow 2^L\ge84.5 \Rightarrow L=7$ (receptive field $1+2\cdot127=255$).

```python
rf_a = 1 + 1 * (1 + 2 + 4 + 8 + 16 + 32)
layers_b = 7
```

</details>

### Exercise 5 · Pinball loss
*✍️ By hand · ★☆☆*

For the 0.1-quantile, compute the pinball loss when the truth is 50 and the forecast is (a) 40 and (b) 60. Which mistake is punished more, and why does that make sense for a 10th percentile?

In [ ]:
loss_a = None
loss_b = None
check('(a) forecast 40', loss_a, 1.0); check('(b) forecast 60', loss_b, 9.0)

<details>
<summary><b>💡 Hint</b></summary>

$\rho_\tau(u)=\max(\tau u,(\tau-1)u)$ with $u=y-\hat y$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $u=10$: $\max(1, -9)=1$. (b) $u=-10$: $\max(-1, 9)=9$. Over-forecasting is punished 9× more, so the model is pushed down until only ~10 % of outcomes fall below its forecast, which is the definition of the 10th percentile.

```python
loss_a = max(0.1 * 10, (0.1 - 1) * 10)
loss_b = max(0.1 * -10, (0.1 - 1) * -10)
```

</details>

### Exercise 6 · How many training windows?
*✍️ By hand · ★☆☆*

A series has $T=1000$ points. With lookback $L=168$ and horizon $H=24$ and stride 1, how many (input, target) windows can you cut? If you split by time and keep the last 200 points for testing, how many training windows are there whose **targets** lie entirely in the first 800 points?

In [ ]:
n_windows = None
n_train = None
check('all windows', n_windows, 809); check('training windows', n_train, 609)

<details>
<summary><b>💡 Hint</b></summary>

A window needs $L+H$ consecutive points.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Windows: $T-L-H+1=1000-192+1=809$. Training windows inside the first 800 points: $800-192+1=609$.

```python
n_windows = 1000 - 168 - 24 + 1
n_train = 800 - 168 - 24 + 1
```

</details>

### Exercise 7 · Instance normalisation (RevIN)
*✍️ By hand · ★★☆*

An input window is $[10, 12, 14, 12]$. RevIN normalises each window by its own mean and standard deviation (population std), the model predicts in normalised units, and the output is de-normalised with the same statistics. (a) What is the normalised window? (b) The model outputs $[1.0, -0.5]$; what is the forecast in original units?

In [ ]:
normed = None
forecast = None
check('(a) normalised window', normed, [-np.sqrt(2), 0, np.sqrt(2), 0], tol=1e-4); check('(b) de-normalised forecast', forecast, [12 + np.sqrt(2), 12 - 0.5 * np.sqrt(2)], tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

Mean 12; population std $=\sqrt{(4+0+4+0)/4}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Mean $12$, std $\sqrt{2}\approx1.414$. (a) $[-1.414, 0, 1.414, 0]$. (b) $12+1.414\cdot[1.0,-0.5]=[13.414, 11.293]$. Per-window scaling lets one global model handle series with very different levels (1 unit vs 10,000 units sold) and reduces distribution shift over time.

```python
normed = (np.array([10, 12, 14, 12]) - 12) / np.sqrt(2)
forecast = 12 + np.sqrt(2) * np.array([1.0, -0.5])
```

</details>

## Part C · Code from scratch

### Exercise 8 · Sliding windows
*💻 Code · ★☆☆*

Implement `make_windows(series, L, H)` returning `X` of shape `(n, L)` and `Y` of shape `(n, H)` for every window with stride 1 (window $i$: inputs `series[i:i+L]`, targets `series[i+L:i+L+H]`). Use `np.lib.stride_tricks.sliding_window_view` or plain indexing.

In [ ]:
def make_windows(series, L, H):
    return None, None
X_, Y_ = make_windows(np.arange(10.), 4, 2)
check('number of windows', None if X_ is None else len(X_), 5)
check('first window', None if X_ is None else np.r_[X_[0], Y_[0]], [0, 1, 2, 3, 4, 5])
check('last window targets', None if Y_ is None else Y_[-1], [8, 9])

<details>
<summary><b>💡 Hint</b></summary>

`sliding_window_view(series, L + H)` gives all windows of length L+H; split the columns.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Shuffling these windows into train/validation would leak the future (overlapping windows share points), so split by time first and cut windows afterwards.

```python
def make_windows(series, L, H):
    W = np.lib.stride_tricks.sliding_window_view(series, L + H)
    return W[:, :L].copy(), W[:, L:].copy()
```

</details>

### Exercise 9 · Causal dilated convolution
*💻 Code · ★★☆*

Implement `causal_conv1d(x, w, dilation)` for a 1-D input: $y_t=\sum_{i=0}^{k-1}w_i\,x_{t-d\cdot i}$, treating $x_{t<0}=0$ (left zero-padding), so the output has the same length as the input and **never uses future values**.

In [ ]:
def causal_conv1d(x, w, dilation):
    return None
x_ = np.arange(1., 9.)
check('dilation 1, kernel [1, 1] = x_t + x_{t-1}', causal_conv1d(x_, np.array([1., 1.]), 1), [1, 3, 5, 7, 9, 11, 13, 15])
check('dilation 2', causal_conv1d(x_, np.array([1., -1.]), 2), [1, 2, 2, 2, 2, 2, 2, 2])
y1 = causal_conv1d(x_, np.array([0.5, 0.3, 0.2]), 2); x2 = x_.copy(); x2[5:] = 100
y2 = causal_conv1d(x2, np.array([0.5, 0.3, 0.2]), 2)
check('no peeking at the future', None if y1 is None else y1[:5], None if y2 is None else y2[:5])

<details>
<summary><b>💡 Hint</b></summary>

Pad `x` on the left with `(k-1)*dilation` zeros; then `y[t] = sum(w[i] * xp[t + pad - d*i])`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Causality is what makes a TCN valid for forecasting; dilation is what makes its receptive field grow exponentially with depth.

```python
def causal_conv1d(x, w, dilation):
    k = len(w); pad = (k - 1) * dilation
    xp = np.r_[np.zeros(pad), x]
    return np.array([sum(w[i] * xp[t + pad - dilation * i] for i in range(k)) for t in range(len(x))])
```

</details>

### Exercise 10 · Pinball loss finds quantiles
*💻 Code · ★★☆*

Implement `pinball(y, q, tau)` = mean pinball loss of a constant forecast `q` for observations `y`. Then show numerically that the `q` minimising it over a fine grid is (close to) the empirical `tau`-quantile of `y`, for `tau` = 0.1, 0.5 and 0.9.

In [ ]:
def pinball(y, q, tau):
    return None
y_ = rng.gamma(2.0, 3.0, 2000)
if pinball(y_, 1.0, 0.5) is not None:
    grid = np.linspace(y_.min(), y_.max(), 2001)
    best = {t: grid[np.argmin([pinball(y_, q, t) for q in grid])] for t in (0.1, 0.5, 0.9)}
else: best = {t: None for t in (0.1, 0.5, 0.9)}
check('tau=0.5 → median', best[0.5], np.quantile(y_, 0.5), tol=0.1)
check('tau=0.1 → 10th percentile', best[0.1], np.quantile(y_, 0.1), tol=0.1)
check('tau=0.9 → 90th percentile', best[0.9], np.quantile(y_, 0.9), tol=0.1)

<details>
<summary><b>💡 Hint</b></summary>

`u = y - q`; `np.maximum(tau * u, (tau - 1) * u).mean()`. Search `q` over `np.linspace(y.min(), y.max(), 2001)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

This is why training a network with pinball loss for several τ gives prediction intervals: each output head learns a quantile of the conditional distribution.

```python
def pinball(y, q, tau):
    u = y - q
    return float(np.mean(np.maximum(tau * u, (tau - 1) * u)))
```

</details>

### Exercise 11 · A DLinear-style forecaster
*💻 Code · ★★☆*

DLinear splits each input window into a **trend** (moving average) and a **remainder**, and maps each with its own linear layer to the horizon. Implement `moving_average(X, k)` (row-wise, length-preserving, edge-padded with the first/last value, odd `k`) and `fit_dlinear(X, Y, k)` returning a `predict(X)` function, fitting both linear maps jointly by least squares on `[trend | remainder | 1]`.

In [ ]:
def moving_average(X, k):
    return None

def fit_dlinear(X, Y, k=25):
    return None
t_ = np.arange(3000); s_ = 0.01 * t_ + 3 * np.sin(2 * np.pi * t_ / 24) + rng.normal(0, 0.5, len(t_))
Xw, Yw = np.lib.stride_tricks.sliding_window_view(s_, 96 + 24)[:, :96], np.lib.stride_tricks.sliding_window_view(s_, 96 + 24)[:, 96:]
check('moving average of a constant row', moving_average(np.full((2, 9), 5.0), 5), np.full((2, 9), 5.0))
check('moving average length-preserving', None if moving_average(Xw[:3], 25) is None else moving_average(Xw[:3], 25).shape, (3, 96))
pred = fit_dlinear(Xw[:2000], Yw[:2000], 25)
mae_dl = None if pred is None else np.mean(np.abs(pred(Xw[2000:]) - Yw[2000:]))
mae_naive = np.mean(np.abs(Xw[2000:, -24:] - Yw[2000:]))
check('DLinear beats seasonal naive', None if mae_dl is None else bool(mae_dl < mae_naive), True)

<details>
<summary><b>💡 Hint</b></summary>

Pad each row with `k//2` copies of its first and last values, then use `np.convolve(row, np.ones(k)/k, 'valid')`. Solve `np.linalg.lstsq(np.c_[T, R, ones], Y)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

On many benchmarks this tiny model is a strong baseline. Here it should beat the seasonal naive forecast on a noisy seasonal series with trend.

```python
def moving_average(X, k):
    p = k // 2
    Xp = np.concatenate([np.repeat(X[:, :1], p, 1), X, np.repeat(X[:, -1:], p, 1)], axis=1)
    return np.array([np.convolve(row, np.ones(k) / k, mode="valid") for row in Xp])

def fit_dlinear(X, Y, k=25):
    def features(X):
        T = moving_average(X, k)
        return np.c_[T, X - T, np.ones(len(X))]
    W = np.linalg.lstsq(features(X), Y, rcond=None)[0]
    return lambda Xn: features(Xn) @ W
```

</details>

### Exercise 12 · N-BEATS residual stacking
*💻 Code · ★★★*

An N-BEATS block outputs a **backcast** (its explanation of the input) and a **forecast**; the next block sees the residual. Implement `nbeats_forward(x, blocks)` where each block is a function `x → (backcast, forecast)`: return the total forecast $\sum_\ell \hat y_\ell$ and the final residual. Test it with *linear* blocks: a 'trend block' that fits a straight line through the input's 24-step moving average and extrapolates it, followed by a 'seasonal block' that repeats the last 24 values.

In [ ]:
def nbeats_forward(x, blocks):
    return None, None
H = 24; tt = np.arange(96)
def trend_block(x):                      # line through the 24-step moving average (removes the daily cycle exactly)
    ma = np.convolve(x, np.ones(24) / 24, mode="valid"); a, b0 = np.polyfit(np.arange(len(ma)) + 11.5, ma, 1)
    return a * np.arange(len(x)) + b0, a * np.arange(len(x), len(x) + H) + b0
def season_block(x): return x, np.tile(x[-24:], H // 24)
x_in = 0.2 * tt + 2 * np.sin(2 * np.pi * tt / 24)
fc, res = nbeats_forward(x_in, [trend_block, season_block])
truth = 0.2 * np.arange(96, 120) + 2 * np.sin(2 * np.pi * np.arange(96, 120) / 24)
check('trend + season forecast', fc, truth, tol=1e-6)
check('residual fully explained', None if res is None else float(np.abs(res).max()), 0.0, tol=1e-9)

<details>
<summary><b>💡 Hint</b></summary>

Loop: `b, f = block(residual)`; `residual = residual - b`; `total += f`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The trend block explains the slope, the seasonal block models what's left: an interpretable decomposition, the idea behind N-BEATS' interpretable configuration. In the real model each block is a small MLP trained end to end.

```python
def nbeats_forward(x, blocks):
    residual, total = x.astype(float).copy(), 0.0
    for block in blocks:
        b, f = block(residual)
        residual = residual - b; total = total + f
    return total, residual
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Deep Learning for Time Series](Deep%20Learning%20for%20Time%20Series.md).*